# Run v4 on the LIARArg test set (n=425)Parses the 425 LIARArg stratified test rows with the v4 adapter and writes predictions to `phase2_data_liar/parser_preds_v4.jsonl`. These predictions feed into the Phase 1 pipeline for the v4 integration result in Table 4.

In [ ]:
%%bash
echo "=== is the LIARArg parse actually running? ==="
ps -p $(pgrep -f "v4_liararg_parse\|parse_v4" | head -1) -o etime,pid 2>/dev/null \
    || echo "no LIARArg parse process running"
echo ""
echo "=== LIARArg parse log (if it started) ==="
tail -10 ~/argument-aware-rag/eval_logs/v4_liararg_parse.log 2>/dev/null \
    || echo "no v4_liararg_parse.log yet"
echo ""
echo "=== how many predictions on disk? ==="
wc -l ~/argument-aware-rag/phase2_data_liar/parser_preds_v4.jsonl 2>/dev/null \
    || echo "no v4 predictions file yet"

In [ ]:
%%bash
python3 -c "
import json
total, empty = 0, 0
with open('phase2_data_liar/parser_preds_v4.jsonl') as f:
    for line in f:
        rec = json.loads(line)
        total += 1
        p = rec['prediction']
        if len(p['claim_components'])==0 and len(p['premise_components'])==0:
            empty += 1
print(f'53 partial predictions: {total} total, {empty} empty ({100*empty/total:.0f}%)')
"

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate

# Kill any lingering processes
pkill -f v4_liararg_parse 2>/dev/null || true

nohup python3 -u <<'PY' > eval_logs/v4_liararg_parse.log 2>&1 &
"""v4 LIARArg parse — forces KV cache on at generate() to avoid the
O(N²) generation cost the previous attempt hit."""
import sys, json, time, os, torch
sys.path.insert(0, '.')
from pathlib import Path
from src.phase2.config import load_phase2_config
from src.phase2.student import build_student, _format_input, _parse_output

cfg = load_phase2_config('configs/phase2_beta_qwen1.5b_lora_v4.yaml')
cfg.student.max_input_len = 2048
cfg.student.max_target_len = 2048

student = build_student(cfg.student)
student.load(cfg.student_output_dir)
tok = student._tokenizer
model = student._model

# CRITICAL: force use_cache at every level of the PEFT wrapper
model.config.use_cache = True
if hasattr(model, "base_model"):
    model.base_model.config.use_cache = True
if hasattr(model, "get_base_model"):
    try: model.get_base_model().config.use_cache = True
    except: pass

print(f"[v4-parse] use_cache: model={model.config.use_cache}", flush=True)

OUT = Path('phase2_data_liar/parser_preds_v4.jsonl')
done_ids = set()
if OUT.exists():
    with open(OUT) as f:
        for line in f:
            try: done_ids.add(int(json.loads(line)['row_id']))
            except: pass
print(f"[v4-parse] resuming — {len(done_ids)} done", flush=True)

with open('data/test.jsonl') as f:
    rows = [json.loads(l) for l in f if l.strip()]

t0 = time.time()
n_done = 0
with open(OUT, 'a') as fout:
    for row in rows:
        rid = int(row['id'])
        if rid in done_ids: continue
        text = row.get('full_text') or row.get('summary') or row.get('statement', '')
        try:
            inp = _format_input(text)
            prompt_str = tok.apply_chat_template(
                [{"role": "user", "content": inp}],
                tokenize=False, add_generation_prompt=True,
            )
            enc = tok([prompt_str], return_tensors='pt', truncation=True,
                      max_length=cfg.student.max_input_len).to(student._device)
            with torch.no_grad():
                out = model.generate(
                    **enc,
                    max_new_tokens=cfg.student.max_target_len,
                    do_sample=False,
                    use_cache=True,   # ← explicit cache override
                    pad_token_id=tok.pad_token_id,
                    eos_token_id=tok.eos_token_id,
                )
            input_len = enc['input_ids'].shape[-1]
            raw = tok.decode(out[0][input_len:], skip_special_tokens=True)
            pred, reasoning = _parse_output(raw)
        except Exception as e:
            pred = {'claim_components':[],'premise_components':[],'citation_components':[],'relations':[]}
            reasoning = f"[error: {e}]"
        fout.write(json.dumps({
            'row_id': rid, 'prediction': pred, 'reasoning': reasoning
        }, ensure_ascii=False) + '\n')
        fout.flush(); os.fsync(fout.fileno())
        n_done += 1
        if n_done % 10 == 0:
            elapsed = time.time() - t0
            rate = n_done / max(elapsed, 1e-6)
            eta = (len(rows) - n_done - len(done_ids)) / max(rate, 1e-6) / 60
            print(f"  {n_done + len(done_ids)}/{len(rows)}  ({elapsed:.0f}s, {rate:.2f}/s, ETA {eta:.1f}m)", flush=True)

total, empty = 0, 0
with open(OUT) as f:
    for line in f:
        rec = json.loads(line)
        total += 1
        p = rec['prediction']
        if len(p['claim_components'])==0 and len(p['premise_components'])==0:
            empty += 1
print(f"\n[v4-parse] DONE. {total} rows, {empty} empty ({100*empty/total:.0f}%)", flush=True)
PY
echo "PID: $!"
sleep 5
tail -5 eval_logs/v4_liararg_parse.log